Goal:
Construct out first dummy classifier and baseline model.

Plan Steps:
* Load the data
* preprocess
* dummy classifier (all images are good) + evaluation
* First simple baseline model... feature extraction using Resnet-18, and using Nearest Neighbors to build embeddings models for each product type.
  - ResNet-18 needs RGB, 224 x 224 as input images
  - Convert images to RGB, resize to 224×224, and apply ResNet-18’s expected normalization.
  - Use a frozen pretrained ResNet-18 to extract one feature vector per image.
  - Build a separate Nearest Neighbours model for each product using only normal reference images.
  - Use average distance to the _k_ nearest normal examples as the anomaly score.
  - Select the defect threshold from held-out normal validation data—not the test set.
  - Evaluate once on the test set using per-product AUROC, macro-average AUROC, recall, specificity, and balanced accuracy.
  - Expected limitation: resizing may miss very small defects.

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
from anomaly_detection import MVTecImageDataset, build_mvtec_manifest
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, average_precision_score, roc_auc_score
from torchvision import transforms
from torchvision.models import ResNet18_Weights
from torch.utils.data import DataLoader
from torch import nn
from torchvision.models import resnet18
import torch
import torch.nn.functional as F
from sklearn.neighbors import NearestNeighbors

In [3]:
data_path = Path.home() / 'projects/anomaly-detection/data/mvtec/'

In [4]:
df = build_mvtec_manifest(data_path)

In [5]:
def get_product_train_test(df, product):
    available_products = df['product'].unique().tolist()
    if product not in available_products:
        raise ValueError(
            f"{product!r} is invalid. Choose from: {sorted(available_products)}"
            )
    train_df = df[(df['product'] == product) & (df['split'] == 'train') & (~df['is_anomaly'])].copy()
    test_df = df[(df['product'] == product) & (df['split'] == 'test')].copy()
    return train_df.reset_index(drop=True), test_df.reset_index(drop=True)

In [6]:
normal_pool_df, test_df = get_product_train_test(df, 'bottle')

In [7]:
reference_df, validation_df = train_test_split(normal_pool_df, test_size=0.2, random_state=17)

In [8]:
normal_pool_df.is_anomaly.value_counts()

is_anomaly
False    209
Name: count, dtype: int64

In [9]:
y_true = test_df['is_anomaly'].to_numpy(dtype=int)
y_pred_dummy = np.zeros_like(y_true)

In [10]:
print(pd.crosstab(y_true, y_pred_dummy, colnames=['Predictions'], rownames=['True']))

Predictions   0
True           
0            20
1            63


In [11]:
print(classification_report(y_true, y_pred_dummy, zero_division=0))

              precision    recall  f1-score   support

           0       0.24      1.00      0.39        20
           1       0.00      0.00      0.00        63

    accuracy                           0.24        83
   macro avg       0.12      0.50      0.19        83
weighted avg       0.06      0.24      0.09        83



The 24% accuracy hides the important fact that detector misses every defect

## Baseline Model with ResNet-18

### Loading and preprocessing

#### Here, we'll be using pytorch + ResNet-18 specs to resize each image, convert it to a tensor, and normalize it according to ImageNet dataset.  


In [12]:
resnet_weights = ResNet18_Weights.DEFAULT
weights_transform = resnet_weights.transforms()
weights_transform

ImageClassification(
    crop_size=[224]
    resize_size=[256]
    mean=[0.485, 0.456, 0.406]
    std=[0.229, 0.224, 0.225]
    interpolation=InterpolationMode.BILINEAR
)

#### Since we saw some defects were touching edges of the images, we will customize the preprocessing to only resize images without cropping them.

In [13]:
resnet_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=weights_transform.mean,
        std=weights_transform.std,
    ),
])

resnet_transform

Compose(
    Resize(size=(224, 224), interpolation=bilinear, max_size=None, antialias=True)
    ToTensor()
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
)

In [14]:
# MVTecImageDataset is shared by the modelling notebooks. It returns
# (image_tensor, anomaly_label, path) for each manifest row.

In [15]:
reference_dataset = MVTecImageDataset(
    reference_df,
    transform=resnet_transform,
)

In [16]:
image_tensor, label, path = reference_dataset[0]

In [17]:
type(image_tensor)

torch.Tensor

In [18]:
print("Dataset size:", len(reference_dataset))
print("Tensor shape:", image_tensor.shape)
print("Tensor type:", image_tensor.dtype)
print("Label:", label)
print("Path:", path)
print("Minimum:", image_tensor.min().item())
print("Maximum:", image_tensor.max().item())

Dataset size: 167
Tensor shape: torch.Size([3, 224, 224])
Tensor type: torch.float32
Label: 0
Path: /home/ali/projects/anomaly-detection/data/mvtec/bottle/train/good/156.png
Minimum: -1.621286153793335
Maximum: 2.640000104904175


In [19]:
validation_dataset = MVTecImageDataset(
    validation_df,
    transform=resnet_transform,
)

test_dataset = MVTecImageDataset(
    test_df,
    transform=resnet_transform,
)

#### Creating a Dataloader that batches images

In [20]:

reference_loader = DataLoader(
    reference_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
)

In [21]:
batch_images, batch_labels, batch_paths = next(iter(reference_loader))

In [22]:
batch_images.shape

torch.Size([32, 3, 224, 224])

In [23]:
batch_labels

tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0])

In [24]:
batch_paths[:2]

('/home/ali/projects/anomaly-detection/data/mvtec/bottle/train/good/156.png',
 '/home/ali/projects/anomaly-detection/data/mvtec/bottle/train/good/132.png')

In [25]:
print("Images:", batch_images.shape)
print("Labels:", batch_labels.shape)
print("Paths:", len(batch_paths))
print("Unique labels:", batch_labels.unique())
print("Number of batches:", len(reference_loader))

Images: torch.Size([32, 3, 224, 224])
Labels: torch.Size([32])
Paths: 32
Unique labels: tensor([0])
Number of batches: 6


In [26]:
reference_df.shape[0]/32

5.21875

## Feature extraction

### Load the ResNet-18 model with ImageNet pre-trained weights

In [27]:
model = resnet18(weights=resnet_weights)

print(model.fc)

model.fc = nn.Identity()

Linear(in_features=512, out_features=1000, bias=True)


In [28]:
# Prevent the parameters from changing (freeze the pre-trained weights)
# for parameter in model.parameters():
#     parameter.requires_grad = False

# This is equivalent, single line
model.requires_grad_(False)

# Make layers behave in inference mode
model.eval()

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_sta

##### Note: ResNet-18 archeticture (Grouped Blocks):
```text
   1. conv1: Initial 7 × 7 convolution; detects basic visual patterns.
   * bn1: Batch-normalizes the first convolution’s activations.
   * relu: Applies a nonlinear activation.
   * maxpool: Reduces spatial resolution.
   2. layer1.block1: Two 3 × 3 convolutions producing 64-channel features.
   4. layer1.block2: Two 3 × 3 convolutions maintaining 64-channel features.
   6. layer2.block1: Two 3 × 3 convolutions downsampling to 128-channel features.
   8. layer2.block2: Two 3 × 3 convolutions maintaining 128-channel features.
   10. layer3.block1: Two 3 × 3 convolutions downsampling to 256-channel features.
   12. layer3.block2: Two 3 × 3 convolutions maintaining 256-channel features.
   14. layer4.block1: Two 3 × 3 convolutions downsampling to 512-channel features.
   16. layer4.block2: Two 3 × 3 convolutions maintaining 512-channel features.
   * avgpool: Averages spatial information into one value per channel.
   18. fc: Final fully connected classification layer; maps features to output classes. (REPLACED WITH Idendity to output the features not classes)
```


### Extract features

#### a small test

In [29]:
with torch.inference_mode():
    batch_features = model(batch_images)

print(batch_features.shape)

torch.Size([32, 512])


Now each image has a feature-rich 512 embedding/vector/tensor

In [30]:
batch_features[0].shape

torch.Size([512])

#### Now lets use the GPU if available


In [31]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = model.to(device)


In [32]:
torch.cuda.is_available()


True

In [33]:
def extract_features(model, loader, device):
    feature_batches = []
    label_batches =[]
    paths = []
    
    # Although it has been called before, there is no harm here, just to ensure it runs in inference mode
    model.eval()
    
    with torch.inference_mode():
        for images, labels, batch_paths in loader:
            images = images.to(device) # move the images to the GPU if available
            features = model(images) # extract the features (i.e run predict)

            feature_batches.append(features.cpu())
            label_batches.append(labels)
            paths.extend(batch_paths)

    all_features = torch.cat(feature_batches, dim=0)
    all_labels = torch.cat(label_batches, dim=0)

    return all_features, all_labels, paths

In [34]:
reference_features, reference_labels, reference_paths = extract_features(model, reference_loader, device)
validation_features, validation_labels, validation_paths = extract_features(model, validation_loader, device)
test_features, test_labels, test_paths = extract_features(model, test_loader, device)

In [35]:
test_labels

tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])

In [36]:
reference_features.device

device(type='cpu')

In [37]:
reference_features.shape

torch.Size([167, 512])

In [38]:
validation_features.shape

torch.Size([42, 512])

In [39]:
test_features.shape

torch.Size([83, 512])

### Normalize the feature embeddings before using them in the nearest neighbor model

In [40]:
reference_features_normalized = F.normalize(
    reference_features, p=2, dim=1
)
validation_features_normalized = F.normalize(
    validation_features, p=2, dim=1
)
test_features_normalized = F.normalize(
    test_features, p=2, dim=1
)

In [41]:
torch.linalg.vector_norm(validation_features_normalized[0])

tensor(1.)

#### Fit the nearest neighbor model

In [42]:
k = 5

neighbor_model = NearestNeighbors(
    n_neighbors=k,
    metric="euclidean",
)

neighbor_model.fit(reference_features_normalized)

,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance<https://docs.scipy.org/doc/scipy/reference/spatial.distance.html>`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'euclidean'
,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",5
,"radius radius: float, default=1.0Range of parameter space to use by default for :meth:`radius_neighbors`queries.",1.0
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"p p: float (positive), default=2Parameter for the Minkowski metric fromsklearn.metrics.pairwise.pairwise_distances. When p = 1, this isequivalent to using manhattan_distance (l1), and euclidean_distance(l2) for p = 2. For arbitrary p, minkowski_distance (l_p) is used.",2
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
Name,Type,Value
effective_metric_ effective_metric_: strMetric used to compute distances to neighbors.,str,'eu...an'
effective_metric_params_ effective_metric_params_: dictParameters for the metric used to compute distances to neighbors.,dict,{}


In [43]:
validation_distances, validation_neighbor_indices = neighbor_model.kneighbors(validation_features_normalized)

In [44]:
reference_distances, reference_neighbor_indices = neighbor_model.kneighbors(reference_features_normalized)

In [45]:
validation_distances.shape

(42, 5)

In [46]:
validation_neighbor_indices

array([[152, 103,  92, 129,  10],
       [  7, 136,  59, 134, 105],
       [132,  91,   9,  67,  59],
       [160, 119, 123,  10, 107],
       [120,  89, 127, 118,  75],
       [152,  10, 119,  33,  98],
       [ 90, 111,  74, 150,  92],
       [144, 147,  69, 115,  97],
       [ 73,  16,  19, 119,  78],
       [ 52, 125,  36,  41, 119],
       [ 53,  76,  99,  81,  73],
       [ 10, 119, 145,  14, 114],
       [ 34,  12,  42, 149, 141],
       [ 80, 123, 153,  48,  73],
       [ 20, 149, 122, 103,  12],
       [ 10, 104, 114,  18, 145],
       [156, 103, 112,  27, 140],
       [ 77, 126,  78, 156, 110],
       [ 19,  89, 162,  95,  99],
       [ 62,  32,  34,  39,  31],
       [120, 165,  99, 130,  76],
       [ 20, 138,   8, 137, 144],
       [  7, 134, 163,   5,  16],
       [  4,  48,  89,  80,  79],
       [102,  75,  58,  98,  18],
       [ 20, 138, 144,   8,  18],
       [ 33,  62,   7, 102,  98],
       [  7,  41,  59, 148,  11],
       [129, 110,  70,  60,   4],
       [ 59,  

In [47]:
reference_scores = reference_distances.mean(axis=1)
print(reference_distances.shape)
print(reference_neighbor_indices.shape)
print(reference_scores.shape)
print(reference_scores.min())
print(reference_scores.mean())
print(reference_scores.max())

(167, 5)
(167, 5)
(167,)
0.07396617531776428
0.0970098388642296
0.14304703720935308


In [48]:
validation_scores = validation_distances.mean(axis=1)
print(validation_distances.shape)
print(validation_neighbor_indices.shape)
print(validation_scores.shape)
print(validation_scores.min())
print(validation_scores.mean())
print(validation_scores.max())

(42, 5)
(42, 5)
(42,)
0.10369573831558228
0.12160361524493919
0.2004410743713379


In [49]:
validation_neighbor_indices.max()

np.int64(165)

In [50]:
reference_distances.shape

(167, 5)

In [51]:
validation_neighbor_indices

array([[152, 103,  92, 129,  10],
       [  7, 136,  59, 134, 105],
       [132,  91,   9,  67,  59],
       [160, 119, 123,  10, 107],
       [120,  89, 127, 118,  75],
       [152,  10, 119,  33,  98],
       [ 90, 111,  74, 150,  92],
       [144, 147,  69, 115,  97],
       [ 73,  16,  19, 119,  78],
       [ 52, 125,  36,  41, 119],
       [ 53,  76,  99,  81,  73],
       [ 10, 119, 145,  14, 114],
       [ 34,  12,  42, 149, 141],
       [ 80, 123, 153,  48,  73],
       [ 20, 149, 122, 103,  12],
       [ 10, 104, 114,  18, 145],
       [156, 103, 112,  27, 140],
       [ 77, 126,  78, 156, 110],
       [ 19,  89, 162,  95,  99],
       [ 62,  32,  34,  39,  31],
       [120, 165,  99, 130,  76],
       [ 20, 138,   8, 137, 144],
       [  7, 134, 163,   5,  16],
       [  4,  48,  89,  80,  79],
       [102,  75,  58,  98,  18],
       [ 20, 138, 144,   8,  18],
       [ 33,  62,   7, 102,  98],
       [  7,  41,  59, 148,  11],
       [129, 110,  70,  60,   4],
       [ 59,  

#### Lets pick a threashold based on the distribution of the validation scores,  the 95th percentile!

In [55]:
threshold_percentile = 95

threshold = np.percentile(validation_scores,threshold_percentile)

print("The 95th percentile Threshold:", threshold)

The 95th percentile Threshold: 0.15631639212369913


In [56]:
validation_predictions = (validation_scores > threshold).astype(int)

# Since the validation dataset is all defect free images, we care about false positives given this threshold:
validation_false_positive_rate = validation_predictions.mean()

print("Validation false-positive rate:", validation_false_positive_rate)

Validation false-positive rate: 0.07142857142857142


#### Evaluate the test split

In [52]:
test_distances, test_neighbor_indices = neighbor_model.kneighbors(test_features_normalized)

In [53]:
test_scores = test_distances.mean(axis=1)
print(test_distances.shape)
print(test_neighbor_indices.shape)
print(test_scores.shape)
print(test_scores.min())
print(test_scores.mean())
print(test_scores.max())

(83, 5)
(83, 5)
(83,)
0.10263057798147202
0.24611084427101068
0.48562620878219603


In [54]:
test_labels

tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])

In [57]:
test_predictions = (test_scores > threshold).astype(int)

In [58]:
print(classification_report(test_labels, test_predictions))

              precision    recall  f1-score   support

           0       0.83      0.95      0.88        20
           1       0.98      0.94      0.96        63

    accuracy                           0.94        83
   macro avg       0.90      0.94      0.92        83
weighted avg       0.95      0.94      0.94        83



In [59]:
roc_auc = roc_auc_score(test_labels, test_scores)
pr_auc = average_precision_score(test_labels, test_scores)
print('roc_auc:', roc_auc)
print('pr_auc:',pr_auc)

roc_auc: 0.988095238095238
pr_auc: 0.9960150240858752


In [62]:
test_df.is_anomaly.value_counts()

is_anomaly
True     63
False    20
Name: count, dtype: int64

### This is only for the bottle category... the baseline model correctly identified 59 out of 63 defect images, and missed 4. It also gave a single false alarm out of 20 defect free images